In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/hotel_bookings.csv")

df['arrival_date'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'] + '-' +
    df['arrival_date_day_of_month'].astype(str)
)

daily =(
    df.groupby('arrival_date')
    .size()
    .reset_index(name='y')   
    
)


daily = daily.rename(columns={'arrival_date': 'ds'})
daily = daily.sort_values('ds').reset_index(drop=True)

daily.head()


,ds,y
0,2015-07-01,122
1,2015-07-02,93
2,2015-07-03,56
3,2015-07-04,88
4,2015-07-05,53


In [2]:
train_size = int(len(daily) * 0.8)
train_df = daily.iloc[:train_size]
test_df = daily.iloc[train_size:]

In [3]:
test_df.head()

,ds,y
634,2017-03-26,126
635,2017-03-27,183
636,2017-03-28,88
637,2017-03-29,154
638,2017-03-30,179


In [4]:
from prophet import Prophet
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    
)

model.fit(train_df)

Importing plotly failed. Interactive plots will not work.
21:14:51 - cmdstanpy - INFO - Chain [1] start processing
21:14:52 - cmdstanpy - INFO - Chain [1] done processing


In [5]:
from prophet import Prophet
tuned_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.15
    
)

tuned_model.fit(train_df)

21:14:52 - cmdstanpy - INFO - Chain [1] start processing
21:14:52 - cmdstanpy - INFO - Chain [1] done processing


In [6]:
future = test_df[['ds']]
forecast = model.predict(future)

forecast[['ds','yhat']].head()

,ds,yhat
0,2017-03-26,158.280010
1,2017-03-27,197.437896
2,2017-03-28,161.894489
3,2017-03-29,181.549368
4,2017-03-30,209.688266


In [7]:
future = test_df[['ds']]
forecast_tuned = tuned_model.predict(future)

forecast_tuned[['ds','yhat']].head()

,ds,yhat
0,2017-03-26,148.603029
1,2017-03-27,191.933675
2,2017-03-28,151.491945
3,2017-03-29,172.274723
4,2017-03-30,206.595947


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_true = test_df['y'].values
y_pred = forecast['yhat'].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

mae , rmse

(38.945807703395744, np.float64(46.95308203942526))

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_true = test_df['y'].values
y_pred_tuned = forecast_tuned['yhat'].values

mae_tuned = mean_absolute_error(y_true, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_true, y_pred_tuned))

mae_tuned, rmse_tuned


(35.740923532771745, np.float64(44.566996782561176))